# Simplified Project Score Model

## Design Philosophy

This model is a distilled version of a comprehensive project scoring system, reduced to 6 core parameters that capture what actually matters:

| Code | Dimension | Core question |
|---|---|---|
| D1 | Scope & Clarity | Is the work clearly defined enough to execute well? |
| D2 | Commercial Value | Will the project create healthy financial value? |
| D3 | Delivery Capacity | Can the current team actually deliver it well? |
| D4 | Execution Risk | Can it be delivered without schedule collapse or operational firefighting? |
| D5 | Client & Engagement | Is the client responsive, reliable, and workable? |
| D6 | Strategic Growth | Does this help the company grow beyond the immediate project? |


## Final Score Equation
$$
Score = G \times \left(D1 + D2 + D3 + D4 + D5 + D6 + B_{rev} + B_{ltv} + B_{strat}\right)
$$
where:
$$
G \in \{0,1\}
$$
is the product of all hard gates. If any gate fails, the final project score becomes zero.



In [37]:
d = {
    # =========================
    # Hard gates (each is 0 or 1; 0 means STOP)
    # =========================

    "gcontract": 1,  # Is there a signed contract/approval? [0=no, 1=yes]
    "glegal": 1,     # Cleared legal and ethical checks? [0=no, 1=yes]
    "gclient": 1,    # Client identity and payment path verified? [0=risky, 1=trusted]
    "gfeasible": 1,  # Can we actually build/deliver this? [0=no, 1=yes]
    "gready": 1,     # Are we ready to start now? [0=no, 1=yes]

    # =========================
    # D1 - Scope & Clarity
    # =========================

    "rd": 0.85,      # How well are requirements documented? [0-1, higher=clearer]
    "ac": 0.90,      # How clearly is "done" defined? [0-1, higher=clearer]
    "am": 0.15,      # How unclear/ambiguous is the scope? [0-1, lower=better]
    "fc": 0.80,      # How well is feature creep controlled? [0-1, higher=better]
    "sc": 2,         # Number of scope changes since kickoff. [0+, lower=more stable]

    # =========================
    # D2 - Commercial Value
    # =========================

    "v": 2500000,    # Total deal value/revenue. [>=0, higher=more revenue]
    "c": 1600000,    # Cost to deliver. [>=0, lower=better margin]
    "cm": 1.00,      # How well do payments match agreed terms? [0-1, higher=better]
    "pp": 0.25,      # How much of the project is complete? [0-1, higher=more done]
    "bs": 400000,    # Budget spent so far. [>=0, higher=more used]
    "tb": 1800000,   # Total approved budget. [>=0, higher=more available]
    "ep": 0.70,      # Chance of future expansion. [0-1, higher=more upside]

    # =========================
    # D3 - Delivery Capacity
    # =========================

    "ta": 0.90,      # How much team bandwidth is free? [0-1, higher=more available]
    "sk": 0.80,      # How well do team skills fit this work? [0-1, higher=better fit]
    "ld": 0.85,      # How ready is the delivery lead? [0-1, higher=better]
    "kc": 0.30,      # Reliance on a few key people. [0-1, lower=safer]
    "bf": 0.60,      # Backup/bench coverage strength. [0-1, higher=more coverage]

    # =========================
    # D4 - Execution Risk
    # =========================

    "tr": 90,        # Timeline requested by client, in days. [>0, higher=more time given]
    "te": 100,       # Timeline we estimate we need, in days. [>0, higher=more effort needed]
    "mc": 4,         # Milestones finished on time. [0+, higher=better progress]
    "mt": 12,        # Total number of milestones. [1+, higher=more checkpoints]
    "ri": 8,         # Number of risks identified. [0+, higher=more known complexity]
    "rm": 7,         # Number of risks mitigated. [0 to ri, higher=better control]
    "rr": 2,         # Number of risks that actually occurred. [0+, lower=better]
    "db": 1,         # Number of blocked dependencies. [0+, lower=better flow]
    "de": 5,         # Total number of dependencies. [0+, higher=more coordination]
    "qc": 0.75,      # Confidence in delivery quality. [0-1, higher=better]

    # =========================
    # D5 - Client & Engagement
    # =========================

    "rt": 6.0,       # Average client reply time, in hours. [>=0, lower=better]
    "coc": 0.80,     # How clear is communication? [0-1, higher=better]
    "py": 1.00,      # How reliably does the client pay? [0-1, higher=safer]
    "cs": 0.85,      # How satisfied is the client? [0-1, higher=better]
    "dma": 1.00,     # How easily can we reach the decision-maker? [0-1, higher=faster decisions]
    "ng": 1,         # Number of negative signals from client. [0+, lower=healthier]

    # =========================
    # D6 - Strategic Growth
    # =========================

    "sv": 0.75,      # How well does this fit our business goals? [0-1, higher=better fit]
    "bp": 0.80,      # Brand/reputation benefit. [0-1, higher=stronger]
    "rp": 0.75,      # Chance of repeat business. [0-1, higher=better]
    "ut": 0.60,      # Potential to upsell. [0-1, higher=more opportunity]
    "wh": 0.65,      # Chance of winning/keeping this deal. [0-1, higher=better]
    "qs": 0.70,      # Quality of the lead/source. [0-1, higher=more reliable]

    # =========================
    # Bonus layer
    # =========================

    "ltv": 7500000,  # Long-term value of this client. [>=0, higher=better]
    "csp": 0.70,     # Value as a case study/marketing asset. [0-1, higher=better]
    "vta": 0.60,     # Fit with our target industry/vertical. [0-1, higher=better]
    "rf": 0.50,      # Likelihood of generating referrals. [0-1, higher=better]
}


## Hard Gates

Hard gates are **non-negotiable pass/fail filters**. These are not soft scoring inputs.  
If any one of these fails, the project should not proceed regardless of upside.

| Gate | Description |
|---|---|
| G1 | Signed contract exists, or there is a credible approval path |
| G2 | Legal, ethical, and sanctions clearance is confirmed |
| G3 | Client is verified and has a legitimate payment path |
| G4 | The project is technically feasible with current or near-term capability |
| G5 | Minimum execution readiness exists: accountable owner, kickoff path, and operational readiness |

The gate function is:

$$
G = G_1 \times G_2 \times G_3 \times G_4 \times G_5
$$

If any gate equals 0, then:

$$
Score = 0
$$

This makes the model operationally realistic: some projects should be rejected immediately, not merely scored slightly lower.


In [24]:
import math
from pprint import pprint

EPS = 1e-9

def clamp(x, low=0.0, high=1.0):
    return max(low, min(x, high))

def evaluate_gates(d):
    gates = [
        int(d["gcontract"]),
        int(d["glegal"]),
        int(d["gclient"]),
        int(d["gfeasible"]),
        int(d["gready"]),
    ]
    G = 1
    for g in gates:
        G *= g  
    return G

## D1 - Scope & Clarity

This dimension measures whether the work is sufficiently defined to execute cleanly.

Poor scope definition is one of the most common reasons projects fail.  
This dimension therefore remains separate and explicit.

### Inputs

| Variable | Meaning | Range |
|---|---|---|
| `rd` | Requirements documented and agreed | 0 to 1 |
| `ac` | Acceptance criteria defined | 0 to 1 |
| `am` | Ambiguity index, where lower is better | 0 to 1 |
| `fc` | Feature-creep protection / change control discipline | 0 to 1 |
| `sc` | Number of scope changes after kickoff | integer, 0 or more |

### Formula

First, define the quality blend:

$$
Q_{scope} = 0.30 \cdot rd + 0.25 \cdot ac + 0.25 \cdot (1-am) + 0.20 \cdot fc
$$

Then define the scope instability penalty:

$$
M_{scope} = \frac{1}{1 + 0.2 \cdot sc}
$$

Final dimension score:

$$
D1 = 20 \cdot Q_{scope} \cdot M_{scope}
$$

### Interpretation

- The weighted blend evaluates **how clearly the project is defined**.
- The decay term penalizes **post-kickoff instability**.
- Each scope change reduces score progressively rather than absolutely.

This gives the model real sensitivity to scope volatility.


In [25]:
def d1_scope(d):
    rd = clamp(d["rd"])
    ac = clamp(d["ac"])
    am = clamp(d["am"])
    fc = clamp(d["fc"])
    sc = max(0, d["sc"])

    quality = (
        0.30 * rd
        + 0.25 * ac
        + 0.25 * (1 - am)
        + 0.20 * fc
    )
    scope_decay = 1 / (1 + 0.2 * sc)
    return 20 * quality * scope_decay

## D2 - Commercial Value

This dimension captures whether the project is financially healthy.

### Inputs

| Variable | Meaning | Range |
|---|---|---|
| `v` | Project revenue / contract value | positive currency |
| `c` | Estimated delivery cost | positive currency |
| `cm` | Client payment compliance / punctuality | 0 to 1 |
| `pp` | Physical progress / percent complete | 0 to 1 |
| `bs` | Budget spent to date | non-negative currency |
| `tb` | Total approved budget | positive currency |
| `ep` | Expansion potential | 0 to 1 |

### Financial Sub-metrics

Margin:

$$
margin = \max\left(0,\frac{v-c}{v+\epsilon}\right)
$$

Cost Performance Index proxy:

$$
CPI = \frac{pp \cdot tb}{bs+\epsilon}
$$

### Dimension Formula

$$
D2 = 25 \cdot \left[
0.35 \cdot \frac{margin}{0.4}
+ 0.25 \cdot CPI
+ 0.20 \cdot cm
+ 0.20 \cdot ep
\right]
$$

### Interpretation

- CPI is capped at **1.3** to avoid distorted rewards from low-spend situations.
- Payment behaviour matters because revenue quality is not just deal size.
- Expansion potential adds a forward-looking commercial signal.


In [26]:
def d2_commercial(d):
    v = max(EPS, d["v"])
    c = max(0, d["c"])
    cm = clamp(d["cm"])
    pp = clamp(d["pp"])
    bs = max(0, d["bs"])
    tb = max(EPS, d["tb"])
    ep = clamp(d["ep"])

    margin = max(0.0, (v - c) / (v + EPS))
    cpi = (pp * tb) / (bs + EPS)
    cpi_n = min(cpi, 1.3) / 1.3

    score = (
        0.35 * min(margin / 0.4, 1.0)
        + 0.25 * cpi_n
        + 0.20 * cm
        + 0.20 * ep
    )
    return 25 * score


## D3 - Delivery Capacity

This dimension measures whether the team can realistically deliver the project.

### Inputs

| Variable | Meaning | Range |
|---|---|---|
| `ta` | Team availability ratio (allocated / required) | 0 to 1+ |
| `sk` | Skill and knowledge fit | 0 to 1 |
| `ld` | Lead or PM readiness | 0 to 1 |
| `kc` | Key-person concentration, where lower is better | 0 to 1 |
| `bf` | Bench strength / backup depth | 0 to 1 |

### Resilience Term

$$
resilience = 0.5 \cdot (1-kc) + 0.5 \cdot bf
$$

### Dimension Formula

$$
D3 = 20 \cdot \left[
0.35 \cdot \min(ta,1)
+ 0.30 \cdot sk
+ 0.15 \cdot ld
+ 0.20 \cdot resilience
\right]
$$

### Interpretation

- Team availability is capped at 1 because overstaffing should not create artificial bonus points.
- The resilience term rewards **distributed capability** and **backup coverage**.
- This dimension keeps delivery scoring grounded in human execution reality.


In [27]:
def d3_delivery(d):
    ta = max(0, d["ta"])
    sk = clamp(d["sk"])
    ld = clamp(d["ld"])
    kc = clamp(d["kc"])
    bf = clamp(d["bf"])
    resilience = 0.5 * (1 - kc) + 0.5 * bf
    score = (
        0.35 * min(ta, 1.0)
        + 0.30 * sk
        + 0.15 * ld
        + 0.20 * resilience
    )
    return 20 * score

## D4 - Execution Risk

This dimension measures all the risks that coming during the execution of the project.

### Inputs

| Variable | Meaning | Range |
|---|---|---|
| `tr` | Client-requested timeline in days | positive integer |
| `te` | Internal realistic estimate in days | positive integer |
| `mc` | Milestones completed on time | integer |
| `mt` | Total milestones planned | integer |
| `ri` | Risks identified | integer |
| `rm` | Risks with mitigation plan | integer |
| `rr` | Risks that materialized | integer |
| `db` | Dependencies currently blocked | integer |
| `de` | Total external dependencies | integer |
| `qc` | Quality confidence / review confidence | 0 to 1 |

### Sub-metrics

Schedule realism:

$$
pressure = \min\left(\frac{tr}{te+\epsilon}, 1\right)
$$

Schedule performance:

$$
SPI = \frac{mc}{mt+\epsilon}
$$

Risk preparedness:

$$
prep = \frac{rm+1}{ri+1}
$$

Risk materialization penalty:

$$
mat\_penalty = \frac{1}{1 + e^{5\left(\frac{rr}{ri+1} - 0.3\right)}}
$$

Dependency health:

$$
dep\_health = 1 - \frac{db}{de+1}
$$

### Dimension Formula

$$
D4 = 20 \cdot \left[
0.20 \cdot pressure
+ 0.15 \cdot SPI
+ 0.20 \cdot prep
+ 0.15 \cdot mat\_penalty
+ 0.15 \cdot dep\_health
+ 0.15 \cdot qc
\right]
$$

### Interpretation

- `pressure` measures whether the client's timeline is realistic.
- `SPI` captures actual delivery rhythm.
- `prep` rewards mitigation discipline.
- `mat_penalty` is a **sigmoid penalty**, which is powerful because it penalizes sharply once risk materialization becomes meaningfully high.
- `dep_health` captures operational dependency blockage.

This dimension is one of the strongest parts of the hybrid model because it turns execution difficulty into a mathematically sensitive score.


In [28]:
def d4_execution_risk(d):
    tr = max(0, d["tr"])
    te = max(EPS, d["te"])
    mc = max(0, d["mc"])
    mt = max(0, d["mt"])
    ri = max(0, d["ri"])
    rm = max(0, d["rm"])
    rr = max(0, d["rr"])
    db = max(0, d["db"])
    de = max(0, d["de"])
    qc = clamp(d["qc"])

    pressure = min(tr / (te + EPS), 1.0)
    spi = mc / (mt + EPS)

    prep = (rm + 1) / (ri + 1)
    mat_ratio = rr / (ri + 1)
    mat_penalty = 1 / (1 + math.exp(5 * (mat_ratio - 0.3)))
    dep_health = 1 - (db / (de + 1))

    dep_health = clamp(dep_health)
    prep = max(0.0, prep)
    spi = clamp(spi, 0.0, 1.5)  
    spi = min(spi, 1.0)

    score = (
        0.20 * pressure
        + 0.15 * spi
        + 0.20 * prep
        + 0.15 * mat_penalty
        + 0.15 * dep_health
        + 0.15 * qc
    )
    return 20 * score

## D5 - Client & Engagement

This dimension evaluates whether the client is practically workable.

The most important mathematical feature here is **exponential response-time decay**.

Why exponential decay?

Because client responsiveness is not linear in real project operations:

- a 2-hour response is much better than a 24-hour response
- a 24-hour response is much better than a 72-hour response
- delays compound operational friction non-linearly

### Inputs

| Variable | Meaning | Range |
|---|---|---|
| `rt` | Average response time in hours | non-negative |
| `coc` | Communication clarity | 0 to 1 |
| `py` | Payment reliability / creditworthiness | 0 to 1 |
| `cs` | Client satisfaction / CSAT | 0 to 1 |
| `dma` | Decision-maker accessibility | 0 to 1 |
| `ng` | Negative engagement signals | integer, 0 or more |

### Sub-metrics

Response quality:

$$
resp = e^{-0.04 \cdot rt}
$$

Negative signal damper:

$$
damper = \max(0, 1 - 0.15 \cdot ng)
$$

### Dimension Formula

$$
D5 = 20 \cdot damper \cdot \left[
0.30 \cdot resp
+ 0.20 \cdot coc
+ 0.20 \cdot py
+ 0.20 \cdot cs
+ 0.10 \cdot dma
\right]
$$

### Interpretation

- Fast client response produces a steep operational advantage.
- The damper reduces the entire dimension when there are repeated negative signals such as ghosting, evasion, or unreliable participation.
- This captures client quality much more realistically than a flat subjective rating.


In [29]:
def d5_client(d):
    rt = max(0, d["rt"])
    coc = clamp(d["coc"])
    py = clamp(d["py"])
    cs = clamp(d["cs"])
    dma = clamp(d["dma"])
    ng = max(0, d["ng"])
    resp = math.exp(-0.04 * rt)
    damper = max(0.0, 1 - 0.15 * ng)
    score = (
        0.30 * resp
        + 0.20 * coc
        + 0.20 * py
        + 0.20 * cs
        + 0.10 * dma
    )
    return 20 * damper * score

## D6 - Strategic Growth

This dimension captures the long-term leverage value of the project.

A project should not be evaluated only by immediate delivery and immediate cash value.  
Some projects help the company grow through:

- strategic positioning
- stronger brand association
- repeat business
- upsell opportunity
- higher win quality
- stronger referral pathways

### Inputs

| Variable | Meaning | Range |
|---|---|---|
| `sv` | Strategic fit / portfolio value | 0 to 1 |
| `bp` | Brand value of the client | 0 to 1 |
| `rp` | Repeat-business probability | 0 to 1 |
| `ut` | Upsell potential | 0 to 1 |
| `wh` | Win probability / close confidence | 0 to 1 |
| `qs` | Source quality / referral quality | 0 to 1 |

### Formula

$$
D6 = 25 \cdot \left[
0.25 \cdot sv
+ 0.20 \cdot bp
+ 0.20 \cdot rp
+ 0.15 \cdot ut
+ 0.10 \cdot wh
+ 0.10 \cdot qs
\right]
$$

### Interpretation

This is the bounded strategic layer.

It gives strong scores to projects that improve the company's long-term position, but the truly unbounded growth effect is added later through the bonus layer.


In [30]:
def d6_strategic(d):
    sv = clamp(d["sv"])
    bp = clamp(d["bp"])
    rp = clamp(d["rp"])
    ut = clamp(d["ut"])
    wh = clamp(d["wh"])
    qs = clamp(d["qs"])
    score = (
        0.25 * sv
        + 0.20 * bp
        + 0.20 * rp
        + 0.15 * ut
        + 0.10 * wh
        + 0.10 * qs
    )
    return 25 * score

## Unbounded Bonus Layer

The six core dimensions are intentionally bounded and interpretable.

The model becomes **unbounded upward** through three additive bonus terms:

- revenue scale bonus
- lifetime value bonus
- strategic leverage bonus

These bonuses sit on top of the core score.

### Revenue Bonus

$$
B_{rev} = 8 \cdot \ln\left(1 + \frac{v}{100000}\right)
$$

This rewards larger deals, but because the function is logarithmic, very large values do not overwhelm operational fundamentals.

### Lifetime Value Bonus

$$
B_{ltv} = 6 \cdot \ln\left(1 + \frac{ltv}{v+\epsilon}\right)
$$

This rewards accounts whose future value is much larger than the first deal.

### Strategic Leverage Bonus

$$
B_{strat} = 5 \cdot \frac{csp + vta + rf}{3}
$$

where:

| Variable | Meaning | Range |
|---|---|---|
| `csp` | Case-study potential | 0 to 1 |
| `vta` | Vertical/thesis alignment | 0 to 1 |
| `rf` | Referral force / network spillover | 0 to 1 |

### Why logarithms?

A logarithmic bonus preserves an important property:

- large projects score higher
- but not so much higher that they automatically dominate weaker but healthier projects

In [31]:
def growth_bonuses(d):
    v = max(EPS, d["v"])
    ltv = max(0, d["ltv"])
    csp = clamp(d["csp"])
    vta = clamp(d["vta"])
    rf = clamp(d["rf"])

    b_rev = 8 * math.log(1 + v / 100000)
    b_ltv = 6 * math.log(1 + ltv / (v + EPS))
    b_strat = 5 * ((csp + vta + rf) / 3)

    return b_rev, b_ltv, b_strat

## Final Model Assembly

The final score is built in four stages:

1. Evaluate hard gates
2. Compute the six core dimensions
3. Compute the three additive bonuses
4. Return the final score and optional breakdown

### Final Equation

$$
Score = G \times \left(D1 + D2 + D3 + D4 + D5 + D6 + B_{rev} + B_{ltv} + B_{strat}\right)
$$

### Why additive scoring?

Additive scoring is used instead of a weighted average because it preserves separation between dimensions:

- a weak dimension remains visibly weak
- a strong strategic project can continue to climb
- score compression is avoided
- the total is more manager-readable in breakdown form

This makes the model more useful for real decision discussions.

In [36]:
G = evaluate_gates(d)

D1 = d1_scope(d)
D2 = d2_commercial(d)
D3 = d3_delivery(d)
D4 = d4_execution_risk(d)
D5 = d5_client(d)
D6 = d6_strategic(d)

B_rev, B_ltv, B_strat = growth_bonuses(d)

core_total = D1 + D2 + D3 + D4 + D5 + D6
bonus_total = B_rev + B_ltv + B_strat
score = G * (core_total + bonus_total)

print(f"Final Score: {score:.2f}")

Final Score: 135.07
